# PyTorch DDP Fashion MNIST Training Example

This example demonstrates how to train a convolutional neural network (CNN) to classify images using the [Fashion MNIST](https://github.com/zalandoresearch/fashion-mnist) dataset and [PyTorch Distributed Data Parallel (DDP)](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html).

This notebook walks you through running that example locally.

## Install the Kubeflow SDK

You need to install the Kubeflow SDK to interact with Kubeflow Trainer APIs:

In [ ]:
!pip install /PATH/TO/LOCAL/SDK

In [53]:
!pip freeze | grep kubeflow

kubeflow @ file:///Users/szaher/go/src/github.com/szaher/sdk/python
kubeflow_trainer_api @ git+https://github.com/kubeflow/trainer.git@ed5f859a24a8b19f8bb7eb09320188050295e713#subdirectory=api/python_api


## Define the Training Function

The first step is to create function to train CNN model using Fashion MNIST data.

In [54]:
def train_fashion_mnist():
    import os

    import torch
    import torch.distributed as dist
    import torch.nn.functional as F
    from torch import nn
    from torch.utils.data import DataLoader, DistributedSampler
    from torchvision import datasets, transforms

    # Define the PyTorch CNN model to be trained
    class Net(nn.Module):
        def __init__(self):
            super(Net, self).__init__()
            self.conv1 = nn.Conv2d(1, 20, 5, 1)
            self.conv2 = nn.Conv2d(20, 50, 5, 1)
            self.fc1 = nn.Linear(4 * 4 * 50, 500)
            self.fc2 = nn.Linear(500, 10)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            x = F.max_pool2d(x, 2, 2)
            x = F.relu(self.conv2(x))
            x = F.max_pool2d(x, 2, 2)
            x = x.view(-1, 4 * 4 * 50)
            x = F.relu(self.fc1(x))
            x = self.fc2(x)
            return F.log_softmax(x, dim=1)

    # Use NCCL if a GPU is available, otherwise use Gloo as communication backend.
    device, backend = ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    print(f"Using Device: {device}, Backend: {backend}")

    # Setup PyTorch distributed.
    local_rank = int(os.getenv("LOCAL_RANK", 0))
    dist.init_process_group(backend=backend)
    print(
        "Distributed Training for WORLD_SIZE: {}, RANK: {}, LOCAL_RANK: {}".format(
            dist.get_world_size(),
            dist.get_rank(),
            local_rank,
        )
    )

    # Create the model and load it into the device.
    device = torch.device(f"{device}:{local_rank}")
    model = nn.parallel.DistributedDataParallel(Net().to(device))
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

    
    # Download FashionMNIST dataset only on local_rank=0 process.
    if local_rank == 0:
        dataset = datasets.FashionMNIST(
            "./data",
            train=True,
            download=True,
            transform=transforms.Compose([transforms.ToTensor()]),
        )
    dist.barrier()
    dataset = datasets.FashionMNIST(
        "./data",
        train=True,
        download=False,
        transform=transforms.Compose([transforms.ToTensor()]),
    )


    # Shard the dataset accross workers.
    train_loader = DataLoader(
        dataset,
        batch_size=100,
        sampler=DistributedSampler(dataset)
    )

    # TODO(astefanutti): add parameters to the training function
    dist.barrier()
    for epoch in range(1, 3):
        model.train()

        # Iterate over mini-batches from the training set
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            # Copy the data to the GPU device if available
            inputs, labels = inputs.to(device), labels.to(device)
            # Forward pass
            outputs = model(inputs)
            loss = F.nll_loss(outputs, labels)
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if batch_idx % 10 == 0 and dist.get_rank() == 0:
                print(
                    "Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}".format(
                        epoch,
                        batch_idx * len(inputs),
                        len(train_loader.dataset),
                        100.0 * batch_idx / len(train_loader),
                        loss.item(),
                    )
                )

    # Wait for the distributed training to complete
    dist.barrier()
    if dist.get_rank() == 0:
        print("Training is finished")

    # Finally clean up PyTorch distributed
    dist.destroy_process_group()

## Local Execution: Train Locally First

To train locally, you would have to pass the name of the mode/backend for the `TrainerClient` to initialize and use for training.

The possible options are: `local`, `docker`, `podman`, `kubernetes` 


In [55]:
from kubeflow.trainer import CustomTrainer, TrainerClient
client = TrainerClient("local")

## List the Training Runtimes

You can get the list of available Training Runtimes to start your TrainJob.

Additionally, it might show available accelerator type and number of available resources.

In [56]:
for runtime in client.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

LocalRuntime(name='torch-distributed', trainer=Trainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework=<Framework.TORCH: 'torch'>, entrypoint=['torchrun'], accelerator='Unknown', accelerator_count='Unknown'), pretrained_model='', create_venv=True, command=['torchrun'], python_path='/var/folders/6q/4crldfcs6fz823gf2ysxgksm0000gn/T/y00e3f58b958-kz98q02c/bin/python', execution_dir='/var/folders/6q/4crldfcs6fz823gf2ysxgksm0000gn/T/y00e3f58b958-kz98q02c')


## Run the Distributed TrainJob

Kubeflow TrainJob will train the above model on 3 PyTorch nodes.

In [57]:
job_name = client.train(
    trainer=CustomTrainer(
        func=train_fashion_mnist,
        # Set how many PyTorch nodes you want to use for distributed training.
        num_nodes=3,
        # Set the resources for each PyTorch node.
        resources_per_node={
            "cpu": 3,
            "memory": "16Gi",
            # Uncomment this to distribute the TrainJob using GPU nodes.
            # "nvidia.com/gpu": 1,
        },
        packages_to_install=[
            "torch==2.7.1",
            "torchvision==0.22.1"
        ]
    ),
    runtime=torch_runtime,
)

## Watch the TrainJob logs

We can use the `get_job_logs()` API to get the TrainJob logs.


In [59]:
_ = client.get_job_logs(job_name, follow=True)

W0709 16:13:22.060000 29089 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Using Device: cpu, Backend: gloo
[W709 16:13:27.902435000 ProcessGroupGloo.cpp:757] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
Distributed Training for WORLD_SIZE: 1, RANK: 0, LOCAL_RANK: 0
Train Epoch: 1 [0/60000 (0%)]	Loss: 2.312570
Train Epoch: 1 [1000/60000 (2%)]	Loss: 2.107793
Train Epoch: 1 [2000/60000 (3%)]	Loss: 1.817873
Train Epoch: 1 [3000/60000 (5%)]	Loss: 1.177836
Train Epoch: 1 [4000/60000 (7%)]	Loss: 0.901831
Train Epoch: 1 [5000/60000 (8%)]	Loss: 0.878027
Train Epoch: 1 [6000/60000 (10%)]	Loss: 0.638893
Train Epoch: 1 [7000/60000 (12%)]	Loss: 0.675583
Train Epoch: 1 [8000/60000 (13%)]	Loss: 0.644498
Train Epoch: 1 [9000/60000 (15%)]	Loss: 0.661601
Train Epoch: 1 [10000/60000 (

## List TrainJobs

List Training Jobs


In [62]:
client.list_jobs()

[]

## Delete the TrainJob

When TrainJob is finished, you can delete the resource.


In [61]:
client.delete_job(job_name)